# 2.3 — Same task, three models
**Session 2.3 · Demo**

What you'll see: one task on Claude Haiku, Sonnet and Opus, with the same prompt and input.
We compare the answers, then what each one cost.

In [1]:
# @title Setup: connect to Claude { display-mode: "form" }
import importlib.util, shutil, subprocess, sys

if importlib.util.find_spec("anthropic") is None:
    installer = (
        ["uv", "pip", "install", "--python", sys.executable]
        if shutil.which("uv")
        else [sys.executable, "-m", "pip", "install"]
    )
    subprocess.check_call([*installer, "-q", "anthropic"])

import html
import json
import os
from getpass import getpass

import anthropic
import pandas as pd
from IPython.display import HTML, display

pd.set_option("display.max_colwidth", None)


def read_api_key():
    """Environment variable, then the Colab secret, then a prompt."""
    key = os.environ.get("ANTHROPIC_API_KEY")
    if key:
        return key
    try:
        from google.colab import userdata

        return userdata.get("ANTHROPIC_API_KEY")
    except Exception:
        return getpass("Paste your Anthropic API key: ")


client = anthropic.Anthropic(api_key=read_api_key())

INK, SOFT_INK, BLUE, GRID = "#0b0b0b", "#52514e", "#2a78d6", "#e1e0d9"


def answer_box(title, text, caption=""):
    """One answer: a title, the text, and a grey caption underneath (tokens, cost, time)."""
    title_html = f'<div style="font-weight:600;color:{INK};margin-bottom:6px">{html.escape(title)}</div>' if title else ""
    caption_html = f'<div style="margin-top:8px;font-size:12px;color:{SOFT_INK}">{html.escape(caption)}</div>' if caption else ""
    return (
        f'<div style="border-left:3px solid {BLUE};background:#fff;padding:8px 12px;margin:4px 0">'
        f"{title_html}"
        f'<div style="white-space:pre-wrap;font-family:monospace;font-size:12.5px;color:{INK}">{html.escape(text)}</div>'
        f"{caption_html}</div>"
    )


def show_answer(text, caption="", title=""):
    display(HTML(answer_box(title, text, caption)))


def show_table(columns, rows, headline=""):
    """A plain table, with an optional headline above it. Cells are HTML."""
    headline_html = f'<div style="font-size:16px;font-weight:600;color:{INK};margin:4px 0 8px">{html.escape(headline)}</div>' if headline else ""
    header = "".join(
        f'<th style="padding:6px 14px 6px 0;text-align:left;color:{SOFT_INK};font-weight:600">{html.escape(name)}</th>'
        for name in columns
    )
    body = "".join(
        "<tr>"
        + "".join(f'<td style="padding:6px 14px 6px 0;vertical-align:top;color:{INK};border-top:1px solid {GRID}">{cell}</td>' for cell in row)
        + "</tr>"
        for row in rows
    )
    display(HTML(f'{headline_html}<table style="border-collapse:collapse;background:#fff"><tr>{header}</tr>{body}</table>'))


ORANGE, AQUA, GREY = "#eb6834", "#1baf7a", "#898781"

# The fields we ask for, and the right answers for the facts.
FIELDS = {
    "initial_term_months": "Initial term (months)",
    "auto_renews": "Renews automatically",
    "renewal_term_months": "Each renewal (months)",
    "non_renewal_notice_days": "Notice to stop renewing (days)",
    "early_exit": "Leaving early",
    "liability_cap": "Liability cap",
    "biggest_risk_for_northgate": "Biggest risk for Northgate",
}
FACTS = {"initial_term_months": 36, "auto_renews": True, "renewal_term_months": 12, "non_renewal_notice_days": 90}


def chip(text, colour):
    return (f'<span style="display:inline-block;padding:1px 8px;border-radius:10px;background:{colour};'
            f'color:#fff;font-size:12px;font-weight:600">{html.escape(text)}</span>')


def read_reply(text):
    """The JSON object in a reply, and how clean the reply was."""
    start, end = text.find("{"), text.rfind("}")
    try:
        data = json.loads(text[start : end + 1])
    except ValueError:
        return {}, chip("not valid JSON", ORANGE)
    if text.strip().startswith("{") and text.strip().endswith("}"):
        return data, chip("plain JSON", AQUA)
    return data, chip("extra text around the JSON", ORANGE)


def is_right(value, expected):
    if isinstance(expected, bool):
        return str(value).strip().lower() in (("true", "yes") if expected else ("false", "no"))
    try:
        return float(value) == expected
    except (TypeError, ValueError):
        return False


def field_cell(key, data):
    if key not in data or data[key] in (None, ""):
        return f'<span style="color:{GREY}">missing</span>'
    value = data[key]
    shown = {True: "yes", False: "no"}.get(value, value) if isinstance(value, bool) else value
    if key in FACTS:
        mark = chip("✓", AQUA) if is_right(value, FACTS[key]) else chip(f"✗ should be {FACTS[key]}", ORANGE)
        return f'<span style="font-size:15px;font-weight:600">{html.escape(str(shown))}</span> {mark}'
    return html.escape(str(shown))


def show_comparison(runs):
    """One column per model, one row per field: facts checked against the clause, then the judgement."""
    replies = {label: read_reply(run["text"]) for label, run in runs.items()}
    cell = f'padding:8px 12px 8px 0;vertical-align:top;border-top:1px solid {GRID};color:{INK}'
    label_cell = f'padding:8px 12px 8px 0;vertical-align:top;border-top:1px solid {GRID};color:{SOFT_INK};width:16%'

    def row(name, cells):
        return f'<tr><td style="{label_cell}">{name}</td>' + "".join(f'<td style="{cell}">{c}</td>' for c in cells) + "</tr>"

    def group(title):
        return (f'<tr><td colspan="{len(runs) + 1}" style="padding:14px 0 4px;color:{BLUE};font-weight:600;'
                f'font-size:13px;letter-spacing:0.04em">{title}</td></tr>')

    rows = group("THE FACTS, CHECKED AGAINST THE CLAUSE")
    rows += "".join(row(FIELDS[key], [field_cell(key, data) for data, _ in replies.values()]) for key in FACTS)
    rows += group("THE JUDGEMENT")
    rows += "".join(
        row(FIELDS[key], [field_cell(key, data) for data, _ in replies.values()]) for key in FIELDS if key not in FACTS
    )
    rows += group("THE REPLY")
    rows += row("Format", [badge for _, badge in replies.values()])
    rows += row("Tokens, cost, time", [
        f'{run["input_tokens"]} in · {run["output_tokens"]} out<br>${run["cost"]:.5f} · {run["seconds"]:.1f}s'
        for run in runs.values()
    ])
    rows += row("Raw reply", [
        f'<details><summary style="cursor:pointer;color:{BLUE}">show</summary>'
        f'<pre style="white-space:pre-wrap;font-size:12px;margin:6px 0 0">{html.escape(run["text"])}</pre></details>'
        for run in runs.values()
    ])
    header = "".join(f'<th style="padding:6px 12px 6px 0;text-align:left;color:{INK};font-size:15px">{html.escape(label)}</th>'
                     for label in runs)
    display(HTML(f'<table style="border-collapse:collapse;background:#fff;width:100%;font-size:14px">'
                 f'<tr><th></th>{header}</tr>{rows}</table>'))


print("Ready.")

Ready.


In [2]:
# @title Answer caching { display-mode: "form" }
# @markdown When this is on, a request that is exactly the same as one already sent gets the saved answer
# @markdown back. Untick it (and run this step) to get fresh answers from Claude.
REPLAY = True  # @param {type:"boolean"}

import hashlib
import json
import os

import anthropic
from anthropic.resources.messages import Messages

REPLAY_DIR = "/content/claude_replay" if os.path.isdir("/content") else os.path.expanduser("~/claude_replay")
os.makedirs(REPLAY_DIR, exist_ok=True)
if not hasattr(Messages, "live_create"):
    Messages.live_create = Messages.create  # The real call, kept so we can switch back.
replay_count = {"saved answers reused": 0, "new calls": 0}


def replay_create(self, **request):
    """Every call to Claude comes through here: reuse a saved answer, or call Claude and save the answer."""
    connection = type(getattr(getattr(self._client, "_client", None), "_transport", None)).__name__
    key = hashlib.sha256(json.dumps([connection, request], sort_keys=True, default=str).encode()).hexdigest()
    path = os.path.join(REPLAY_DIR, f"{key}.json")
    if os.path.exists(path):
        replay_count["saved answers reused"] += 1
        with open(path) as file:
            return anthropic.types.Message.model_validate_json(file.read())
    response = Messages.live_create(self, **request)
    if isinstance(response, anthropic.types.Message):  # Streamed replies aren't saved.
        with open(path, "w") as file:
            file.write(response.model_dump_json())
    replay_count["new calls"] += 1
    return response


Messages.create = replay_create if REPLAY else Messages.live_create
saved = len(os.listdir(REPLAY_DIR))
print(f"Saved answers {'on' if REPLAY else 'off'}. {saved} saved so far in {REPLAY_DIR}.")

Saved answers on. 0 saved so far in /content/claude_replay.


## 1. Task: summarize a contract clause
The clause and the prompt below are the same for every model.

In [3]:
# The contract clause, and the prompt every model gets.
CLAUSE = """9.2 Term and Renewal.
This Agreement commences on the Effective Date and continues
for an initial term of thirty-six (36) months.

Thereafter it renews automatically for successive twelve (12) month periods
unless either party gives written notice of non-renewal not fewer than ninety (90) days
before the end of the then-current term.

Notwithstanding the foregoing, Northgate Logistics may terminate for convenience
after month twenty-four (24) on sixty (60) days' written notice,
subject to an early termination charge equal to three (3) months of the Fees then payable.

The Provider's aggregate liability in any contract year shall not exceed
one hundred and fifty percent (150%) of the Fees paid in that year,
save for breaches of Clause 14 (Data Protection), for which liability shall be uncapped."""

TASK_PROMPT = f"""Summarize this contract clause for a non-lawyer.
Return only a JSON object with these keys:
initial_term_months (number), auto_renews (true or false), renewal_term_months (number),
non_renewal_notice_days (number), early_exit (one sentence), liability_cap (one sentence),
biggest_risk_for_northgate (one sentence).

<clause>
{CLAUSE}
</clause>"""

## 2. Run it on three models
Same prompt, same input. Haiku is the small model, Sonnet the mid-sized one, and Opus the large one.
Each model runs with its default settings, including its own thinking, as you would use it.


In [4]:
# @title Run the task on three models { display-mode: "form" }
import time

MODELS = {
    "Claude Haiku": "claude-haiku-5-5",
    "Claude Sonnet": "claude-sonnet-5-5",
    "Claude Opus": "claude-opus-5-5",
}
PRICES = {  # (input, output) in dollars per million tokens
    "claude-haiku-5-5": (0.10, 0.50),
    "claude-sonnet-5-5": (2.00, 10.00),
    "claude-opus-5-5": (4.00, 20.00),
}


def run_model(prompt, model):
    """Send one prompt and return the answer with its time, tokens and cost."""
    start = time.perf_counter()
    response = client.messages.create(
        model=model, max_tokens=8000, messages=[{"role": "user", "content": prompt}]
    )
    seconds = time.perf_counter() - start

    input_price, output_price = PRICES[model]
    usage = response.usage
    return {
        "text": "".join(
            block.text for block in response.content if block.type == "text"
        ),
        "seconds": seconds,
        "input_tokens": usage.input_tokens,
        "output_tokens": usage.output_tokens,
        "cost": (usage.input_tokens * input_price + usage.output_tokens * output_price)
        / 1_000_000,
    }


runs = {label: run_model(TASK_PROMPT, model) for label, model in MODELS.items()}
show_comparison(runs)

In our test run, all three models got the facts right and kept to the format: for the facts, the small
model is good enough.

Haiku wrote the most output tokens, and was still by far the cheapest.

Haiku contradicts itself in the same reply: its "leaving early" answer is right (the fee applies when
leaving after month 24), but its biggest risk for Northgate says leaving **before** month 24 costs three
months of fees.

### Open-weight model
Qwen 2.5, a small open model (0.5 billion parameters).
Run the step
to add it as a fourth column (untick the box to skip it); the first run downloads about 1 GB.

In [5]:
# @title Optional: the same task on a small open-weight model { display-mode: "form" }
RUN_OPEN_MODEL = True  # @param {type:"boolean"}

if RUN_OPEN_MODEL:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    OPEN_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(OPEN_MODEL)
    open_model = AutoModelForCausalLM.from_pretrained(
        OPEN_MODEL, torch_dtype=torch.float16 if device == "cuda" else torch.float32  # 32-bit is much faster on CPU.
    ).to(device)

    text = tokenizer.apply_chat_template(
        [{"role": "user", "content": TASK_PROMPT}],
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt").to(device)

    start = time.perf_counter()
    output = open_model.generate(**inputs, max_new_tokens=400, do_sample=False)
    seconds = time.perf_counter() - start

    new_tokens = output[0][inputs.input_ids.shape[1] :]
    open_text = tokenizer.decode(new_tokens, skip_special_tokens=True)
    runs["Open-weight (Qwen 0.5B)"] = {
        "text": open_text,
        "seconds": seconds,
        "input_tokens": inputs.input_ids.shape[1],
        "output_tokens": len(new_tokens),
        "cost": 0.0,
    }
    show_comparison(runs)
else:
    show_answer("Switched off. Tick the box and run this step to try it.")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

## 3. The price gap
How much more did the priciest model cost than the cheapest, and what would that mean at volume?

In [6]:
# @title Compare the costs { display-mode: "form" }
claude_runs = {
    label: run for label, run in runs.items() if run["cost"] > 0
}
cheapest = min(claude_runs, key=lambda label: claude_runs[label]["cost"])
priciest = max(claude_runs, key=lambda label: claude_runs[label]["cost"])
ratio = claude_runs[priciest]["cost"] / claude_runs[cheapest]["cost"]

show_table(
    ["model", "this run", "10,000 contracts"],
    [
        [html.escape(label), f"${run['cost']:.5f}", f"<b>${run['cost'] * 10_000:,.2f}</b>"]
        for label, run in claude_runs.items()
    ],
    headline=f"{priciest} cost {ratio:.0f} times more than {cheapest} on this task.",
)

model,this run,"10,000 contracts"
Claude Haiku,$0.00041,$4.10
Claude Sonnet,$0.00361,$36.14
Claude Opus,$0.01119,$111.88


## 4. Model routing: the right model for each request
Section 3 showed that the small model is often good enough, for a fraction of the price. **Routing**
builds on that: send each request to the cheapest model that can handle it.
1. A **router** decides: is this request easy or hard?
2. **Easy** goes to the small model; **hard** goes to the frontier model.

In [7]:
# @title Model routing { display-mode: "form" }
NAVY, ORANGE, SKY = "#2d3e50", "#f5a623", "#aed6f1"

ROUTING_FIGURE = f"""
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 10 820 330" style="max-width:820px;width:100%;
     font-family:Inter,Helvetica,Arial,sans-serif;background:#fff">
  <defs>
    <marker id="arrow" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto">
      <path d="M0,0 L10,5 L0,10 z" fill="{NAVY}"/>
    </marker>
  </defs>

  <!-- Requests -->
  <g fill="#fff" stroke="{NAVY}" stroke-width="1.2">
    <rect x="40" y="88" width="62" height="28" rx="6"/>
    <rect x="48" y="126" width="62" height="28" rx="6"/>
    <rect x="40" y="164" width="62" height="28" rx="6"/>
  </g>
  <g stroke="#b5b3ab" stroke-width="2">
    <line x1="50" y1="98" x2="88" y2="98"/><line x1="50" y1="106" x2="80" y2="106"/>
    <line x1="58" y1="136" x2="96" y2="136"/><line x1="58" y1="144" x2="88" y2="144"/>
    <line x1="50" y1="174" x2="88" y2="174"/><line x1="50" y1="182" x2="80" y2="182"/>
  </g>
  <text x="74" y="214" text-anchor="middle" font-size="13" font-weight="600" fill="{NAVY}">requests</text>

  <!-- Router -->
  <rect x="160" y="112" width="110" height="58" rx="8" fill="{NAVY}"/>
  <text x="215" y="137" text-anchor="middle" font-size="15" font-weight="700" fill="#fff">Router</text>
  <text x="215" y="156" text-anchor="middle" font-size="11" fill="#dfe6ec">easy or hard?</text>

  <!-- Small model -->
  <rect x="340" y="28" width="118" height="54" rx="8" fill="{ORANGE}"/>
  <text x="399" y="52" text-anchor="middle" font-size="14" font-weight="700" fill="#1d1d1d">Small model</text>
  <text x="399" y="70" text-anchor="middle" font-size="11" fill="#1d1d1d">cheap, fast</text>

  <!-- Small model to the answer -->
  <path d="M458,55 C580,55 660,70 690,108" fill="none" stroke="{NAVY}" stroke-width="1.6" marker-end="url(#arrow)"/>

  <!-- Answer -->
  <rect x="640" y="112" width="140" height="58" fill="{SKY}"/>
  <text x="710" y="146" text-anchor="middle" font-size="14" font-weight="700" fill="#1d1d1d">Answer</text>

  <!-- Frontier model -->
  <rect x="455" y="172" width="140" height="58" rx="8" fill="{ORANGE}"/>
  <text x="525" y="197" text-anchor="middle" font-size="14" font-weight="700" fill="#1d1d1d">Frontier model</text>
  <text x="525" y="215" text-anchor="middle" font-size="11" fill="#1d1d1d">expensive, best</text>

  <!-- Fallback -->
  <rect x="460" y="282" width="130" height="38" rx="6" fill="#fff" stroke="{NAVY}" stroke-width="1.4"
        stroke-dasharray="5,4"/>
  <text x="525" y="306" text-anchor="middle" font-size="12.5" font-weight="600" fill="#1d1d1d">Fallback provider</text>

  <!-- Arrows -->
  <line x1="110" y1="141" x2="157" y2="141" stroke="{NAVY}" stroke-width="1.6" marker-end="url(#arrow)"/>
  <line x1="270" y1="125" x2="337" y2="62" stroke="{NAVY}" stroke-width="1.6" marker-end="url(#arrow)"/>
  <text x="268" y="80" text-anchor="end" font-size="12" font-weight="700" fill="{NAVY}">easy</text>
  <text x="268" y="94" text-anchor="end" font-size="10.5" fill="#898781">most traffic</text>
  <line x1="270" y1="158" x2="452" y2="197" stroke="{NAVY}" stroke-width="1.6" marker-end="url(#arrow)"/>
  <text x="360" y="194" font-size="12" font-weight="700" fill="{NAVY}">hard</text>
  <line x1="595" y1="195" x2="637" y2="160" stroke="{NAVY}" stroke-width="1.6" marker-end="url(#arrow)"/>
  <line x1="525" y1="232" x2="525" y2="279" stroke="{NAVY}" stroke-width="1.6" stroke-dasharray="4,3"
        marker-end="url(#arrow)"/>
  <text x="533" y="262" font-size="10.5" fill="#898781">if the provider is down</text>
</svg>
"""
display(HTML(ROUTING_FIGURE))

We don't build the router ourselves. **LiteLLM** is an open-source library that sits between our code and
the model providers. We use its **routing**: it picks a model for each request, from rules we set.

### 4.1 Let LiteLLM route
We give LiteLLM our two models under short names, **small** and **frontier**, and turn on its
**complexity router**. For each question, the router picks the model in three moves:
- it scores the question;
- the score gives a tier: SIMPLE, MEDIUM, COMPLEX or REASONING;
- SIMPLE goes to the small model, everything above it to the frontier model.

The setup follows the Python example in LiteLLM's docs: https://docs.litellm.ai/docs/autorouter_sdk

In [8]:
# @title Set up LiteLLM and its complexity router { display-mode: "form" }
if importlib.util.find_spec("litellm") is None:
    installer = (
        ["uv", "pip", "install", "--python", sys.executable]
        if shutil.which("uv")
        else [sys.executable, "-m", "pip", "install"]
    )
    subprocess.check_call([*installer, "-q", "litellm"])

import litellm
from litellm import Router

litellm.suppress_debug_info = True

KEY = client.api_key
# Our two models, by name
MODEL_LIST = [
    {"model_name": "small", "litellm_params": {"model": "anthropic/claude-haiku-5-5", "api_key": KEY}},
    {"model_name": "frontier", "litellm_params": {"model": "anthropic/claude-opus-5-5", "api_key": KEY}},
    # Another provider is one more entry; nothing else in the code changes. For example:
    # {"model_name": "frontier", "litellm_params": {"model": "bedrock/<Claude model ID on Bedrock>", ...}},
]


async def ask(model_name, question, system=None, **settings):
    """One call through LiteLLM. Returns the text and the full response."""
    messages = [{"role": "user", "content": question}]
    if system:
        messages.insert(0, {"role": "system", "content": system})
    response = await router.acompletion(model=model_name, messages=messages, **settings)
    return response.choices[0].message.content, response


def cost_of(response):
    """LiteLLM's built-in price list: no price table of our own."""
    return litellm.completion_cost(completion_response=response)


TIERS = {"SIMPLE": "small", "MEDIUM": "frontier", "COMPLEX": "frontier", "REASONING": "frontier"}


def make_router(our_rules=()):
    """Our models, plus LiteLLM's complexity router, set up as in the LiteLLM docs."""
    routing_config = {
        "tiers": TIERS,
        "custom_dimensions": list(our_rules),  # Our own keyword rules.
        "max_tokens_from_tier_model": False,  # Keep our max_tokens.
    }
    return Router(
        model_list=[
            *MODEL_LIST,
            {
                "model_name": "auto",
                "litellm_params": {
                    "model": "auto_router/complexity_router",
                    "complexity_router_config": routing_config,
                    "complexity_router_default_model": "frontier",  # If no tier can be decided.
                },
            },
        ]
    )


show_table(["tier", "goes to"], [[tier, name] for tier, name in TIERS.items()])

tier,goes to
SIMPLE,small
MEDIUM,frontier
COMPLEX,frontier
REASONING,frontier


In [9]:
# @title How LiteLLM picks a model { display-mode: "form" }
# The picture: score → tier → name → Claude model, drawn from the settings above.
BOUNDARIES = {"SIMPLE": (-0.2, 0.15), "MEDIUM": (0.15, 0.35), "COMPLEX": (0.35, 0.60), "REASONING": (0.60, 1.0)}
EXAMPLE = ("How many months is the initial term?", -0.15)  # Its score with the built-in rules.
LOOK = {"small": ("#d5f0e5", "#1baf7a"), "frontier": ("#fde3d3", "#eb6834")}


def routing_picture():
    left, width = 170, 690
    x = lambda score: left + (score + 0.2) / 1.2 * width
    ids = {m["model_name"]: m["litellm_params"]["model"].split("/")[-1] for m in MODEL_LIST}
    example_tier = next(t for t, (low, high) in BOUNDARIES.items() if low <= EXAMPLE[1] < high)
    example_name = TIERS[example_tier]
    parts = []
    for y, text in ((95, "1. Score the question"), (205, "2. The tier picks a name"), (305, "3. The name is a model")):
        parts.append(f'<text x="-10" y="{y}" font-size="13" font-weight="600" fill="{SOFT_INK}">{text}</text>')
    for tier, (low, high) in BOUNDARIES.items():
        fill, edge = LOOK[TIERS[tier]]
        on_path = tier == example_tier
        parts.append(f'<rect x="{x(low) + 1}" y="70" width="{x(high) - x(low) - 2}" height="42" rx="4" fill="{fill}" '
                     f'stroke="{BLUE if on_path else edge}" stroke-width="{2.5 if on_path else 1}"/>')
        parts.append(f'<text x="{(x(low) + x(high)) / 2}" y="96" text-anchor="middle" font-size="12" '
                     f'font-weight="700" fill="{INK}">{tier}</text>')
    pin = x(EXAMPLE[1])
    parts.append(f'<path d="M{pin - 7},50 L{pin + 7},50 L{pin},66 z" fill="{BLUE}"/>')
    parts.append(f'<text x="{pin - 10}" y="42" font-size="12" fill="{BLUE}" font-weight="600">'
                 f'"{html.escape(EXAMPLE[0])}" scores {EXAMPLE[1]}</text>')
    for name in dict.fromkeys(TIERS.values()):
        tiers = [t for t in TIERS if TIERS[t] == name]
        centre = (x(BOUNDARIES[tiers[0]][0]) + x(BOUNDARIES[tiers[-1]][1])) / 2
        fill, edge = LOOK[name]
        on_path = name == example_name
        stroke = BLUE if on_path else edge
        for tier in tiers:
            start = (x(BOUNDARIES[tier][0]) + x(BOUNDARIES[tier][1])) / 2
            parts.append(f'<line x1="{start}" y1="113" x2="{centre}" y2="182" stroke="{BLUE if tier == example_tier else GREY}" '
                         f'stroke-width="{2.2 if tier == example_tier else 1.3}" marker-end="url(#{"blue" if tier == example_tier else "grey"})"/>')
        parts.append(f'<rect x="{centre - 70}" y="185" width="140" height="34" rx="17" fill="{fill}" stroke="{stroke}" '
                     f'stroke-width="{2.5 if on_path else 1.4}"/>')
        parts.append(f'<text x="{centre}" y="207" text-anchor="middle" font-size="14" font-weight="700" fill="{INK}">{name}</text>')
        parts.append(f'<line x1="{centre}" y1="220" x2="{centre}" y2="262" stroke="{BLUE if on_path else GREY}" '
                     f'stroke-width="{2.2 if on_path else 1.3}" marker-end="url(#{"blue" if on_path else "grey"})"/>')
        model_id = ids[name]
        family = model_id.split("-")[1].title()
        parts.append(f'<rect x="{centre - 95}" y="265" width="190" height="62" rx="8" fill="#fff" stroke="{stroke}" '
                     f'stroke-width="{2.5 if on_path else 1.4}"/>')
        parts.append(f'<text x="{centre}" y="287" text-anchor="middle" font-size="14" font-weight="700" fill="{INK}">Claude {family}</text>')
        parts.append(f'<text x="{centre}" y="303" text-anchor="middle" font-size="11" font-family="monospace" fill="{SOFT_INK}">{model_id}</text>')
        parts.append(f'<text x="{centre}" y="319" text-anchor="middle" font-size="11" fill="{SOFT_INK}"></text>')
    halo = 'stroke="#fff" stroke-width="4" paint-order="stroke"'
    for value in (0.15, 0.35, 0.60):
        parts.append(f'<text x="{x(value)}" y="128" text-anchor="middle" font-size="11" fill="{SOFT_INK}" {halo}>{value}</text>')
    parts.append(f'<text x="{x(-0.2)}" y="128" font-size="11" fill="{SOFT_INK}" {halo}>simple</text>')
    parts.append(f'<text x="{x(1.0)}" y="128" text-anchor="end" font-size="11" fill="{SOFT_INK}" {halo}>hard</text>')
    markers = "".join(
        f'<marker id="{name}" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto">'
        f'<path d="M0,0 L10,5 L0,10 z" fill="{colour}"/></marker>'
        for name, colour in (("blue", BLUE), ("grey", GREY))
    )
    frames = []
    for top, bottom, label, at, anchor in (
        (22, 258, "LiteLLM's complexity router", 196, "start"),
        (270, 362, "LiteLLM's model list", 876, "end"),
    ):
        frames.append(f'<rect x="180" y="{top}" width="712" height="{bottom - top}" rx="10" fill="none" stroke="{SOFT_INK}" '
                      f'stroke-width="1.2" stroke-dasharray="6,4"/>')
        frames.append(f'<text x="{at}" y="{top + 4}" text-anchor="{anchor}" font-size="12" font-weight="700" fill="{SOFT_INK}" '
                      f'stroke="#fff" stroke-width="6" paint-order="stroke">{label}</text>')
    return (f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 900 372" style="max-width:900px;width:100%;'
            f'font-family:Inter,Helvetica,Arial,sans-serif;background:#fff"><defs>{markers}</defs>{"".join(frames)}'
            f'<g transform="translate(20,26)">{"".join(parts)}</g></svg>')


display(HTML(routing_picture()))


For each question we record LiteLLM's decision: the score, the tier, and the model that answered.

In [10]:
# @title Prepare the routing step { display-mode: "form" }
SYSTEM = f"""<clause>
{CLAUSE}
</clause>

Answer the question about this clause, briefly."""


async def route(question):
    """Send the question to the complexity router: it scores the question and picks the model."""
    text, response = await ask("auto", question, system=SYSTEM, max_tokens=8000)
    headers = response._hidden_params["additional_headers"]  # LiteLLM reports its decision here.
    return {
        "question": question,
        "score": round(float(headers["x-litellm-complexity-router-score"]), 2),
        "tier": headers["x-litellm-complexity-router-tier"],
        "answered by": response.model,
        "answer": text,
        "cost": cost_of(response),
    }


async def route_all(questions):
    return pd.DataFrame([await route(question) for question in questions])


def show_routes(table):
    """The routing results, with each answer folded away under 'answer'."""
    show_table(
        ["question", "score", "tier", "answered by", "cost", ""],
        [
            [
                html.escape(row["question"]),
                f"{row['score']:.2f}",
                f"<b>{row['tier']}</b>",
                html.escape(row["answered by"]),
                f"${row['cost']:.5f}",
                f'<details><summary style="color:{BLUE};cursor:pointer">answer</summary>'
                f'<div style="white-space:pre-wrap;max-width:420px">{html.escape(row["answer"])}</div></details>',
            ]
            for _, row in table.iterrows()
        ],
    )

In [11]:
# The four questions, from a simple lookup to advice.
QUESTIONS = [
    "How many months is the initial term?",
    "How many days before the end of a term must notice of non-renewal be given?",
    "Northgate pays fees of 20,000 euros a month and gives notice to terminate for convenience at the "
    "end of month 30. When does the contract end, and what does leaving early cost in total?",
    "Should Northgate sign this clause as it is? Name the two changes you would negotiate first, and why.",
]

In [12]:
# @title Route the four questions with the built-in rules { display-mode: "form" }
router = make_router()
built_in = await route_all(QUESTIONS)
show_routes(built_in)

question,score,tier,answered by,cost,
How many months is the initial term?,-0.15,SIMPLE,claude-haiku-5-5,$0.00004,answerThe initial term is thirty-six (36) months.
How many days before the end of a term must notice of non-renewal be given?,-0.05,SIMPLE,claude-haiku-5-5,$0.00005,answerNotice of non-renewal must be given not fewer than **ninety (90) days** before the end of the then-current term.
"Northgate pays fees of 20,000 euros a month and gives notice to terminate for convenience at the end of month 30. When does the contract end, and what does leaving early cost in total?",0.01,SIMPLE,claude-haiku-5-5,$0.00036,"answer**End date:** The contract ends about 60 days after the notice, which is roughly the end of **month 32**, not the end of the 36-month initial term. Treating 60 days as two months, notice given at the end of month 30 takes effect at the end of month 32. **Cost of leaving early:** The early termination charge is three months of Fees, so **€60,000** (3 × €20,000). Northgate also pays the normal €20,000 monthly fees for the two months of notice, so total payments are about €640,000 in fees (32 × €20,000) plus the €60,000 charge, roughly **€700,000** overall. The €40,000 for the notice period is part of the regular fees, so the extra cost of early exit is €60,000."
"Should Northgate sign this clause as it is? Name the two changes you would negotiate first, and why.",0.00,SIMPLE,claude-haiku-5-5,$0.00061,"answerNo, Northgate should not sign this as it stands. Two changes come first. 1. **Early termination charge.** Northgate can exit after month 24, but only by paying three months' Fees. That makes the right costly to use, and it sits on top of a 36-month initial term. Northgate should ask for a smaller charge (one month's Fees at most), a step-down that falls to zero by month 30 or 36, or a waiver if the Provider materially breaches. Without this, the exit right is more theoretical than real. 2. **Liability cap.** The 150% cap is measured by ""Fees paid in that year,"" so it could be very small in year one, when onboarding fees may be low. Only data protection breaches are uncapped. Northgate should ask for the cap to be the greater of a fixed sum or 12 months' Fees payable, and for uncapped or higher-cap carve-outs for confidentiality, gross negligence, wilful misconduct, and IP indemnities. A failed logistics service could cause losses far above one year's fees, and the current drafting leaves Northgate exposed to them. Northgate should also check that Clause 14 exists and is correctly cross-referenced, and diarise the 90-day non-renewal deadline."


### 4.2 Teach the router about our requests
We add two rules of our own: words that signal **judgement** and words that signal a **calculation**.
Each rule has two settings:
- its **words**, separated by commas;
- its **weight**: how much it adds to the score when one of its words is in the question.

With a weight of 0.3, a question with one of the words scores about 0.3: above the 0.15 boundary, so it
leaves SIMPLE and goes to the frontier model. Below 0.15, the rule isn't enough on its own.

In [13]:
# @title Add our two rules and route again { display-mode: "form" }
JUDGEMENT_WORDS = "should, negotiate, recommend, advise, why, risk"  # @param {type:"string"}
JUDGEMENT_WEIGHT = 0.3  # @param {type:"slider", min:0.05, max:1, step:0.05}
CALCULATION_WORDS = "total, calculate, how much, cost, costs"  # @param {type:"string"}
CALCULATION_WEIGHT = 0.3  # @param {type:"slider", min:0.05, max:1, step:0.05}


def word_list(words):
    return [word.strip() for word in words.split(",") if word.strip()]


OUR_RULES = [
    {"name": "needsJudgement", "weight": JUDGEMENT_WEIGHT, "keywords": word_list(JUDGEMENT_WORDS)},
    {"name": "needsCalculation", "weight": CALCULATION_WEIGHT, "keywords": word_list(CALCULATION_WORDS)},
]

router = make_router(OUR_RULES)
routed_2 = await route_all(QUESTIONS)
show_routes(routed_2)

question,score,tier,answered by,cost,
How many months is the initial term?,-0.15,SIMPLE,claude-haiku-5-5,$0.00004,answerThe initial term is thirty-six (36) months.
How many days before the end of a term must notice of non-renewal be given?,-0.05,SIMPLE,claude-haiku-5-5,$0.00005,answerNotice of non-renewal must be given not fewer than **ninety (90) days** before the end of the then-current term.
"Northgate pays fees of 20,000 euros a month and gives notice to terminate for convenience at the end of month 30. When does the contract end, and what does leaving early cost in total?",0.32,MEDIUM,claude-opus-5-5,$0.01140,"answer**When it ends:** Termination for convenience is allowed after month 24, so notice at the end of month 30 is valid. With 60 days' notice, the contract ends **60 days after notice is given**, which is roughly the end of month 32. It is exactly 60 days, not exactly two calendar months. **What it costs:** - **Early termination charge:** 3 months × €20,000 = **€60,000** - **Fees during the notice period:** about 2 months × €20,000 = **about €40,000**. These are normal fees for service received, not a penalty. **Total cash out from giving notice to exit:** about **€100,000**, of which €60,000 is the cost of leaving early. By comparison, staying to the end of the initial term (month 36) would cost about €120,000 in fees from month 31 onward. Exiting early therefore saves about €20,000 net."
"Should Northgate sign this clause as it is? Name the two changes you would negotiate first, and why.",0.30,MEDIUM,claude-opus-5-5,$0.02227,"answer**Not as is.** The clause is workable, and the uncapped data-protection liability favours Northgate and should be kept. But two points need fixing first. **1. Fix the liability cap's base.** ""150% of the Fees paid in that year"" ties the Provider's exposure to what has actually been paid in the current contract year. Early in a year, in year one, or if fees are paid in arrears, the cap could be close to zero just when a serious failure happens. - **Ask for:** the greater of a fixed floor amount and 150% of Fees paid *or payable* in the 12 months before the claim. - **Also consider:** making gross negligence, wilful misconduct, fraud and confidentiality breaches uncapped too. **2. Loosen the termination-for-convenience terms.** Northgate is locked in for 24 months. Even then, exiting takes 60 days' notice plus 3 months' Fees, so the practical minimum is about 29 months of spend. The clause is also unclear whether the charge still applies during renewal periods. - **Ask for:** the right to exit earlier (e.g., after month 12). - **Ask for:** a charge that tapers or drops away over time. - **Ask for:** express wording that no charge applies in any renewal term. **Secondary point:** the 90-day non-renewal notice window is long and easy to miss. Push for 30–60 days, or require the Provider to send a renewal reminder."


### 4.3 Compared with sending everything to the frontier model
The same four questions, all on Opus, with no routing.

In [15]:
# @title Send the four questions to Opus, and compare { display-mode: "form" }
frontier_only = 0.0
for question in QUESTIONS:
    _, response = await ask("frontier", question, system=SYSTEM, max_tokens=8000)
    frontier_only += cost_of(response)

show_table(
    ["", "cost of the four questions"],
    [
        ["With routing", f"${built_in['cost'].sum():.5f}"],
        ["Everything on Opus", f"${frontier_only:.5f}"],
    ],
    headline=f"Routing saved {1 - built_in['cost'].sum() / frontier_only:.0%} on these four questions.",
)

,cost of the four questions
With routing,$0.00106
Everything on Opus,$0.03780


### Routing and fallback in the cloud
The big cloud providers now offer routing and fallback too. To route or fall
back **between** providers, you need a gateway like LiteLLM.

| | **AWS** (Amazon Bedrock) | **Microsoft Azure** (AI Foundry + API Management) | **Google Cloud** (Vertex AI) | **LiteLLM** (open source) |
|---|---|---|---|---|
| **Routing:** | Picks a model per request, within **one model family** | A model router picks a model per prompt; traffic can also be split by priority | Picks the best **Gemini** model per request | **Rules you set**, across any models from any provider |
| **Fallback:** | To **another AWS region**, or a fallback model you choose | Stops sending to an endpoint that doesn't respond, and moves to the next one | Mostly **built yourself** | To **any other model or provider** |
| **Switch between providers?** | No | No | No | **Yes** |
